# Notebook 1 — Segment images with Cellpose 4.0.6 and save masks

## Installation 

### for Mac
Latest stable release is Cellpose 4.0.6, [available via conda-forge](https://anaconda.org/conda-forge/cellpose)

```bash
conda env create -f ./envs/cellpose.yml
conda activate cellpose
```

### for colab

`%pip install "cellpose==4.0.6" "torch" "torchvision" "torchaudio" "scikit-image>=0.22.0" "tqdm>=4.66.0" "pandas>=2.2.0"`

In [1]:
# Cell 1 — Imports and Apple Silicon device
from pathlib import Path
from typing import List
import os
import numpy as np
from skimage import io, exposure
from tqdm import tqdm
import torch
from cellpose import models

def has_mps() -> bool:
    return hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

device = torch.device("mps") if has_mps() else torch.device("cpu")
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # safe fallback for rare ops
print("torch:", torch.__version__)
print("device:", device)




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	darwin 
python version: 	3.10.18 
torch version:  	2.7.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


torch: 2.7.1
device: mps


In [2]:
# Cell 2 — Paths and parameters (your project)
project_root = Path("/Users/ashi/github/cm4ai_codefest2025")

# Inputs: data/<channel>/*.jpg  (filenames end with *_<channel>.jpg)
img_root = project_root / "data"
channels: List[str] = ["yellow"]     # segment yellow only (reference masks)

# Outputs
masks_root = project_root / "analysis" / "cellpose_results2"

# Image extensions to search (your data are JPEGs)
image_exts = [".jpg", ".jpeg", ".tif", ".tiff", ".png"]

# Model — stick to cpsam
pretrained_model = "cpsam"

# Segmentation params for faint multiplex RGB
use_auto_diameter = True      # True => diameter=None (auto)
diameter = None if use_auto_diameter else 40.0

# Start extremely permissive to guarantee non-empty masks; tighten later if noisy
cellprob_threshold = -10.0    # raise toward -2.0 → 0.0 if too many pixels included
flow_threshold = 0.0          # raise toward 0.2–0.4 if over-segmentation

invert = False                # try False first; if still empty, set True
batch_size = 4

# Preprocessing
do_rescale_intensity = True   # stretch to full dynamic range per image
do_clahe = False              # set True if rescale alone is insufficient
clahe_clip = 2.0
clahe_tiles = (8, 8)

print("- project root:", project_root)
print("- image root:", img_root)
print("- channels:", channels)
print("- masks root:", masks_root)
print("- model:", pretrained_model)
print("- diameter:", diameter, "(auto)" if use_auto_diameter else "")
print("- thresholds: cellprob", cellprob_threshold, "| flow", flow_threshold)
print("- preprocess: rescale_intensity", do_rescale_intensity, "| clahe", do_clahe)


- project root: /Users/ashi/github/cm4ai_codefest2025
- image root: /Users/ashi/github/cm4ai_codefest2025/data
- channels: ['yellow']
- masks root: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2
- model: cpsam
- diameter: None (auto)
- thresholds: cellprob -10.0 | flow 0.0
- preprocess: rescale_intensity True | clahe False


In [3]:
# Cell 3 — Utilities
def discover_images(folder: Path, exts) -> list[Path]:
    files = []
    for ext in exts:
        files.extend(sorted(folder.glob(f"*{ext}")))
    # de-dup by stem
    seen, uniq = set(), []
    for p in files:
        if p.stem not in seen:
            uniq.append(p); seen.add(p.stem)
    return uniq

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

def preprocess_rgb(img: np.ndarray) -> np.ndarray:
    """Per-image intensity rescue for faint signals."""
    if img.ndim == 2:  # grayscale (unlikely for JPEGs)
        arr = img
        if do_rescale_intensity:
            arr = exposure.rescale_intensity(arr)
        if do_clahe:
            # skimage CLAHE works on [0,1] float; convert back to original dtype
            arrf = exposure.equalize_adapthist(arr, clip_limit=clahe_clip, nbins=256, kernel_size=clahe_tiles)
            arr = (arrf * np.iinfo(img.dtype).max).astype(img.dtype)
        return arr
    # RGB path
    out = img.copy()
    if do_rescale_intensity:
        # rescale each channel independently
        for c in range(out.shape[-1]):
            out[..., c] = exposure.rescale_intensity(out[..., c])
    if do_clahe:
        # CLAHE per channel in float [0,1]
        for c in range(out.shape[-1]):
            outf = exposure.equalize_adapthist(out[..., c], clip_limit=clahe_clip, nbins=256, kernel_size=clahe_tiles)
            out[..., c] = (outf * np.iinfo(img.dtype).max).astype(img.dtype)
    return out


In [4]:
# Cell 4 — Load Cellpose 4.x model (cpsam; MPS via torch device; keep gpu=False)
model = models.CellposeModel(
    gpu=False,
    pretrained_model=pretrained_model,
    device=device
)
print("Loaded model:", model.pretrained_model)


Loaded model: /Users/ashi/.cellpose/models/cpsam


In [5]:
# Cell 5 — Segment YELLOW (RGB → channel_axis=-1), write *_masks.png and keep logs
total_images = 0
failed = []

for ch in channels:  # only "yellow"
    in_dir = img_root / ch
    out_dir = masks_root / ch / "png"
    ensure_dir(out_dir)

    if not in_dir.exists():
        print(f"[SKIP] Missing channel folder: {in_dir}")
        continue

    files = discover_images(in_dir, image_exts)
    if not files:
        print(f"[WARN] No images in {in_dir} with {image_exts}")
        continue

    print(f"[{ch}] {len(files)} images -> {out_dir}")

    for group in tqdm(list(chunked(files, batch_size)), desc=f"Seg {ch}"):
        # read + preprocess
        raw_imgs = [io.imread(p) for p in group]
        imgs = [preprocess_rgb(im) for im in raw_imgs]

        try:
            masks, flows, styles = model.eval(
                imgs,
                diameter=diameter,            # None => auto
                batch_size=len(imgs),
                channel_axis=-1,              # RGB
                invert=invert,
                normalize=False,              # disable tile normalization
                rescale=True,                 # allow Cellpose’s internal rescale
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
            )
        except Exception as e:
            for p in group:
                failed.append((p.name, str(e)))
            continue

        for p, m in zip(group, masks):
            out_path = out_dir / f"{p.stem}_masks.png"   # keeps *_yellow in stem
            io.imsave(out_path, m.astype(np.uint16), check_contrast=False)

    total_images += len(files)

print(f"Done. Segmented {total_images} yellow images.")
print(f"Masks written under: {masks_root}/yellow/png/")
if failed:
    print("Failures:", failed[:5], "... total:", len(failed))


[yellow] 10 images -> /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png


Seg yellow: 100%|██████████| 3/3 [02:59<00:00, 59.81s/it]

Done. Segmented 10 yellow images.
Masks written under: /Users/ashi/github/cm4ai_codefest2025/analysis/cellpose_results2/yellow/png/


new masks look much busier than before — we’ve got thousands of tiny objects segmented!